# RNA-seq Preprocessing — TCGA-BRCA
Produces 8 experimental dataset variants in `statistical_filtered/`.

Notebook location: `Thesis_v3/01_Causal_feature_extraction/RNA/`

| V | Name | Selection logic |
|---|---|---|
| 1 | ultra_conservative | Variance > 98.3 percentile |
| 2 | conservative | Variance > 97.5 percentile |
| 3 | standard | Variance > 95.9 percentile |
| 4 | fdr_significant | Spearman FDR < 0.05, top 1000 by composite |
| 5 | balanced | Var top 25% then Spearman top 10% |
| 6 | correlation | Abs. Spearman > 97.5 percentile |
| 7 | top_correlated | Top 100 positive + top 100 negative Spearman |
| 8 | composite | Average rank of Spearman + MI + Distance corr |


In [2]:
from pathlib import Path

# Notebook: Thesis_v3/01_Causal_feature_extraction/RNA/rna_preprocessing.ipynb
# Path resolution: walk up from cwd until we find the folder containing data/
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "TCGA-BRCA.star_counts.tsv").exists()
)

RNA_FILE     = BASE_DIR / "data" / "TCGA-BRCA.star_counts.tsv"
OUTCOME_FILE = BASE_DIR / "data" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "01_Causal_feature_extraction" / "RNA" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "01_Causal_feature_extraction" / "RNA" / "rna_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_GENES    = 50
TARGET_BROAD = 1000

assert RNA_FILE.exists(),     f"Not found: {RNA_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base   : {BASE_DIR}")
print(f"Output : {OUTPUT_DIR}")
print(f"Cache  : {STATS_CACHE}")
print("Paths OK")


Base   : C:\Users\olegk\Desktop\Thesis_v3
Output : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\RNA\statistical_filtered
Cache  : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\RNA\rna_statistics_cache.csv
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")


In [4]:
print("Loading RNA data...")
rna_raw = pd.read_csv(RNA_FILE, sep="\t", index_col=0)
print(f"  Raw shape  : {rna_raw.shape}  (genes x samples)")

# Transpose to samples x genes
rna = rna_raw.T.copy()

# Extract base Ensembl ID: ENSG00000000003.15|TSPAN6 -> ENSG00000000003.15
rna.columns = [c.split("|")[0] if "|" in c else c for c in rna.columns]

print("Loading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples    : {len(outcome)}")
print(f"  OS.time    : {outcome['OS.time'].min():.0f} - {outcome['OS.time'].max():.0f} days")
print(f"  Events     : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

common  = sorted(set(rna.index) & set(outcome.index))
rna     = rna.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap    : {len(common)} samples")


Loading RNA data...
  Raw shape  : (60660, 1226)  (genes x samples)
Loading outcome...
  Samples    : 1076
  OS.time    : 1 - 8605 days
  Events     : 150 (13.9%)
  Overlap    : 1073 samples


In [5]:
print("Quality control...")

# Remove genes with >20% missing across samples
miss_frac = rna.isna().mean(axis=0)
rna = rna.loc[:, miss_frac <= 0.20]
print(f"  After missing filter (>20%)  : {rna.shape[1]:,} genes")

# Impute residual missing values with per-gene median
imp = SimpleImputer(strategy="median")
rna = pd.DataFrame(imp.fit_transform(rna), index=rna.index, columns=rna.columns)

# Remove zero-variance genes
var = rna.var(axis=0)
rna = rna.loc[:, var > 0]
print(f"  After zero-variance removal  : {rna.shape[1]:,} genes")


Quality control...
  After missing filter (>20%)  : 60,660 genes
  After zero-variance removal  : 58,001 genes


In [6]:
print("Normalization: log2(x+1) then z-score...")

rna_log  = np.log2(rna + 1)
scaler   = StandardScaler()
rna_norm = pd.DataFrame(
    scaler.fit_transform(rna_log),
    index=rna_log.index,
    columns=rna_log.columns
)
print(f"  Mean : {rna_norm.mean().mean():.4f}  (expected ~0)")
print(f"  Std  : {rna_norm.std().mean():.4f}   (expected ~1)")


Normalization: log2(x+1) then z-score...
  Mean : -0.0000  (expected ~0)
  Std  : 1.0005   (expected ~1)


In [7]:
print("Variance pre-filter: keep top 75%...")

var     = rna_norm.var(axis=0)
rna_var = rna_norm.loc[:, var > var.quantile(0.25)]
print(f"  Before : {rna_norm.shape[1]:,}  After : {rna_var.shape[1]:,} genes")


Variance pre-filter: keep top 75%...
  Before : 58,001  After : 43,236 genes


In [8]:
if STATS_CACHE.exists():
    print(f"Loading cached statistics ({STATS_CACHE.name})...")
    stats = pd.read_csv(STATS_CACHE)
    print(f"  Loaded: {len(stats):,} genes")
else:
    import time
    os_time = outcome["OS.time"].values
    genes   = rna_var.columns.tolist()
    n       = len(genes)
    print(f"Computing statistics for {n:,} genes...")

    # 1. Spearman correlation
    print("  [1/3] Spearman correlation...")
    t0 = time.time()
    rho_arr, pval_arr = [], []
    for i, g in enumerate(genes):
        r, p = spearmanr(rna_var[g].values, os_time)
        rho_arr.append(0.0 if np.isnan(r) else float(r))
        pval_arr.append(1.0 if np.isnan(p) else float(p))
        if (i + 1) % 10000 == 0:
            print(f"    {i+1:,} / {n:,}  ({time.time()-t0:.0f}s)")
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")

    # 2. Mutual information
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(
        rna_var.values, os_time, random_state=42, n_neighbors=5
    )

    # 3. Distance correlation (rank-based approximation)
    # Full O(n^2) distance correlation per gene is computationally prohibitive
    # at this scale (~43k genes x 1200 samples). We use Pearson correlation on
    # ranked values as a fast approximation; it captures non-linear monotone
    # associations and contributes ranking diversity to the composite score.
    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = []
    for g in genes:
        x_rank = rankdata(rna_var[g].values).astype(float)
        dc_arr.append(abs(float(np.corrcoef(x_rank, os_rank)[0, 1])))

    stats = pd.DataFrame({
        "gene":          genes,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "variance":      rna_var.var().values,
    })

    # Composite score: average rank across three methods (lower rank = better)
    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (
        stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]
    ) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE.name}")

n_fdr = int((stats["fdr"] < 0.05).sum())
print(f"  FDR < 0.05 : {n_fdr:,} / {len(stats):,} genes")
print(f"  Top 10 by composite score:")
print(stats[["gene","abs_spearman","mutual_info","distance_corr","composite"]].head(10).to_string(index=False))


Computing statistics for 43,236 genes...
  [1/3] Spearman correlation...
    10,000 / 43,236  (14s)
    20,000 / 43,236  (26s)
    30,000 / 43,236  (38s)
    40,000 / 43,236  (53s)
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to rna_statistics_cache.csv
  FDR < 0.05 : 250 / 43,236 genes
  Top 10 by composite score:
              gene  abs_spearman  mutual_info  distance_corr  composite
 ENSG00000204429.4      0.123788     0.065220       0.123788  67.333333
ENSG00000162772.17      0.166550     0.042437       0.166550  88.000000
 ENSG00000272254.1      0.129627     0.044312       0.129627 104.000000
ENSG00000125740.14      0.145466     0.041084       0.145466 118.666667
ENSG00000007908.16      0.115623     0.058607       0.115623 120.000000
ENSG00000058668.15      0.116518     0.053459       0.116518 121.000000
 ENSG00000168497.5      0.127427     0.042833       0.127427 129.333333
ENSG00000142871.18      0.131446     0.039144       0.131446

In [9]:
print("Creating 8 dataset variants...")
print(f"Output: {OUTPUT_DIR}")

def build(gene_list, min_g=MIN_GENES):
    gene_list = [g for g in gene_list if g in rna_var.columns]
    if len(gene_list) < min_g:
        have = set(gene_list)
        gene_list += [g for g in stats["gene"] if g not in have][:min_g - len(gene_list)]
    return rna_var[gene_list]

created = []

# V1 Ultra-Conservative: variance > 98.3 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.983)]["gene"].tolist())
fname = f"rna_1_ultra_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V1","name":"ultra_conservative","n":len(df.columns),"logic":"Variance > 98.3 pct","file":fname})
print(f"  V1 ultra_conservative : {len(df.columns):,} genes")

# V2 Conservative: variance > 97.5 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.975)]["gene"].tolist())
fname = f"rna_2_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V2","name":"conservative","n":len(df.columns),"logic":"Variance > 97.5 pct","file":fname})
print(f"  V2 conservative       : {len(df.columns):,} genes")

# V3 Standard: variance > 95.9 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.959)]["gene"].tolist())
fname = f"rna_3_standard_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V3","name":"standard","n":len(df.columns),"logic":"Variance > 95.9 pct","file":fname})
print(f"  V3 standard           : {len(df.columns):,} genes")

# V4 FDR-significant: Spearman FDR < 0.05, top TARGET_BROAD by composite
fdr_genes = stats[stats["fdr"] < 0.05].head(TARGET_BROAD)["gene"].tolist()
df    = build(fdr_genes, min_g=TARGET_BROAD)
fname = f"rna_4_fdr_significant_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V4","name":"fdr_significant","n":len(df.columns),"logic":"FDR<0.05, top 1000 by composite","file":fname})
print(f"  V4 fdr_significant    : {len(df.columns):,} genes  (raw FDR<0.05: {int((stats['fdr']<0.05).sum())})")

# V5 Balanced: top 25% variance then top 10% Spearman within that subset
var_sub     = stats[stats["variance"] > stats["variance"].quantile(0.75)]
corr_thresh = var_sub["abs_spearman"].quantile(0.90)
df    = build(var_sub[var_sub["abs_spearman"] > corr_thresh]["gene"].tolist())
fname = f"rna_5_balanced_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V5","name":"balanced","n":len(df.columns),"logic":"Var top 25% -> Spearman top 10%","file":fname})
print(f"  V5 balanced           : {len(df.columns):,} genes")

# V6 Correlation-based: abs Spearman > 97.5 percentile
df    = build(stats[stats["abs_spearman"] > stats["abs_spearman"].quantile(0.975)]["gene"].tolist())
fname = f"rna_6_correlation_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V6","name":"correlation","n":len(df.columns),"logic":"Abs Spearman > 97.5 pct","file":fname})
print(f"  V6 correlation        : {len(df.columns):,} genes")

# V7 Top Correlated: top 100 positive + top 100 negative Spearman
top_pos = stats[stats["spearman_rho"] > 0].head(100)["gene"].tolist()
top_neg = stats[stats["spearman_rho"] < 0].tail(100)["gene"].tolist()
df      = build(sorted(set(top_pos + top_neg)))
stats[stats["gene"].isin(set(top_pos + top_neg))][
    ["gene","spearman_rho","pval","fdr","variance"]
].to_csv(OUTPUT_DIR / "rna_7_top_correlated_annotated.csv", index=False)
fname = f"rna_7_top_correlated_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V7","name":"top_correlated","n":len(df.columns),"logic":"Top 100 pos + top 100 neg Spearman","file":fname})
print(f"  V7 top_correlated     : {len(df.columns):,} genes")

# V8 Composite: top TARGET_BROAD by average rank of Spearman + MI + Dcor
df    = build(stats.head(TARGET_BROAD)["gene"].tolist(), min_g=TARGET_BROAD)
fname = f"rna_8_composite_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V8","name":"composite","n":len(df.columns),"logic":"Avg rank Spearman+MI+Dcor, top 1000","file":fname})
print(f"  V8 composite          : {len(df.columns):,} genes")

# Auxiliary files
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "rna_statistics_complete.csv", index=False)
summary = pd.DataFrame(created)
summary.to_csv(OUTPUT_DIR / "datasets_summary.csv", index=False)

print()
print(summary[["V","name","n","logic"]].to_string(index=False))
print(f"Saved to: {OUTPUT_DIR}")


Creating 8 dataset variants...
Output: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\RNA\statistical_filtered
  V1 ultra_conservative : 723 genes
  V2 conservative       : 1,079 genes
  V3 standard           : 1,751 genes
  V4 fdr_significant    : 1,000 genes  (raw FDR<0.05: 250)
  V5 balanced           : 1,066 genes
  V6 correlation        : 1,081 genes
  V7 top_correlated     : 200 genes
  V8 composite          : 1,000 genes

 V               name    n                               logic
V1 ultra_conservative  723                 Variance > 98.3 pct
V2       conservative 1079                 Variance > 97.5 pct
V3           standard 1751                 Variance > 95.9 pct
V4    fdr_significant 1000     FDR<0.05, top 1000 by composite
V5           balanced 1066     Var top 25% -> Spearman top 10%
V6        correlation 1081             Abs Spearman > 97.5 pct
V7     top_correlated  200  Top 100 pos + top 100 neg Spearman
V8          composite 1000 Avg rank Spearman+MI+